# Lab 1 — Inside a Foundation Model

**Course:** Foundation Models: Architecture, Training and Alignment  
**Goal:** understand the basic pipeline: Text → Tokens → Embeddings → Attention → Output.

Run the cells, change the examples, and answer the reflection questions at the end.

In [ ]:
!pip -q install transformers sentence-transformers scikit-learn matplotlib accelerate

## Part A — Tokenization

A model does not see words. It sees token IDs.

In [ ]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")

texts = [
    "Transformers are amazing!",
    "internationalization",
    "The price is 39.99 dollars.",
    "https://www.example.com/foundation-models",
    "שלום עולם",
    "AI 🚀 is changing everything"
]

for text in texts:
    print("
TEXT:", repr(text))
    print("BERT tokens:", bert_tokenizer.tokenize(text))
    print("BERT token count:", len(bert_tokenizer.tokenize(text)))
    print("GPT-2 tokens:", gpt_tokenizer.tokenize(text))
    print("GPT-2 token count:", len(gpt_tokenizer.tokenize(text)))

### Your task
Add three examples of your own: one in Hebrew, one technical sentence, and one with numbers or a URL.

In [ ]:
my_texts = [
    "",  # Hebrew example
    "",  # technical example
    ""   # numbers / URL example
]

for text in my_texts:
    if not text:
        continue
    print("
TEXT:", repr(text))
    print("BERT:", bert_tokenizer.tokenize(text))
    print("GPT-2:", gpt_tokenizer.tokenize(text))

## Part B — Embeddings and Semantic Similarity

Embeddings represent text as vectors. Similar meanings should have similar vectors.

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "A dog is running in the park.",
    "A puppy is playing outside.",
    "The stock market dropped today.",
    "Investors are worried about inflation.",
    "I ate pizza for dinner.",
    "The patient received antibiotics."
]

query = "A small dog is playing."

doc_embeddings = embedding_model.encode(documents)
query_embedding = embedding_model.encode(query)

scores = util.cos_sim(query_embedding, doc_embeddings)[0]

for doc, score in sorted(zip(documents, scores), key=lambda x: x[1], reverse=True):
    print(f"{score:.3f} | {doc}")

### Your task
Create your own mini semantic search dataset with at least 10 short documents from a domain you care about.

In [ ]:
my_documents = [
    # Add at least 10 documents here
]
my_query = ""

if my_documents and my_query:
    my_doc_embeddings = embedding_model.encode(my_documents)
    my_query_embedding = embedding_model.encode(my_query)
    my_scores = util.cos_sim(my_query_embedding, my_doc_embeddings)[0]
    for doc, score in sorted(zip(my_documents, my_scores), key=lambda x: x[1], reverse=True)[:3]:
        print(f"{score:.3f} | {doc}")

## Optional visualization

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

points = PCA(n_components=2).fit_transform(doc_embeddings)
plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1])
for i, doc in enumerate(documents):
    plt.text(points[i, 0] + 0.01, points[i, 1] + 0.01, doc, fontsize=9)
plt.title("Sentence embeddings projected to 2D")
plt.show()

## Part C — Attention Visualization

Attention lets each token attend to other tokens. This visualization is an intuition, not a complete explanation.

In [ ]:
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name, output_attentions=True)

text = "The animal did not cross the street because it was too tired."
inputs = tokenizer(text, return_tensors="pt")
outputs = bert_model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
attention = outputs.attentions[-1][0, 0].detach().numpy()

plt.figure(figsize=(9, 7))
plt.imshow(attention)
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.yticks(range(len(tokens)), tokens)
plt.colorbar()
plt.title("Attention heatmap - last layer, head 0")
plt.show()

### Your task
Change the sentence. Try different heads and layers.

In [ ]:
# Try different layer/head combinations
layer = -1
head = 1

attention = outputs.attentions[layer][0, head].detach().numpy()
plt.figure(figsize=(9, 7))
plt.imshow(attention)
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.yticks(range(len(tokens)), tokens)
plt.colorbar()
plt.title(f"Attention heatmap - layer {layer}, head {head}")
plt.show()

## Part D — GPT vs BERT

GPT-style models generate continuations. BERT-style models fill masked tokens.

In [ ]:
from transformers import pipeline

text_generator = pipeline("text-generation", model="gpt2")
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

print("GPT continuation:")
print(text_generator("Foundation models are", max_new_tokens=30, pad_token_id=50256)[0]["generated_text"])

print("
BERT masked prediction:")
for item in mask_filler("Foundation models are [MASK] for modern AI.")[:5]:
    print(item["token_str"], item["score"])

## Reflection questions

Write short answers:

1. What surprised you about tokenization?
2. What is the difference between semantic similarity and factual correctness?
3. Where did you see the first building block of RAG?
4. Does an attention map prove that a model understands a sentence? Why or why not?
5. What component from this lab could be useful in your final project?

In [ ]:
# Write your answers here as Python comments or replace this cell with Markdown.
# 1.
# 2.
# 3.
# 4.
# 5.